In [1]:
# Dual streaming loader
# Streams all chunked unpivoted weather CSVs into stg_weather_raw

import pymysql
from pathlib import Path
import os

# Masking my Password
import getpass

# Prompt the user for a password
password = getpass.getpass("Enter your password: ")


BASE = Path(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data")
CHUNKS = BASE / "chunks"

def load_chunk(conn, chunk_path):
    sql = f"""
        LOAD DATA LOCAL INFILE '{chunk_path.as_posix()}'
        INTO TABLE stg_weather_raw
        CHARACTER SET utf8mb4
        FIELDS TERMINATED BY ','
        ENCLOSED BY '"'
        LINES TERMINATED BY '\\r\\n'
        IGNORE 1 LINES
        (@source_file,@measure,@datetime_utc,@city_name,@value_text,@value_num)
        SET
          source_file  = @source_file,
          measure      = @measure,
          datetime_utc = NULLIF(TRIM(@datetime_utc),''),
          city_name    = @city_name,
          value_text   = NULLIF(REPLACE(@value_text, '\\r', ''), ''),
          value_num    = NULLIF(REPLACE(@value_num,  '\\r', ''), '');
    """
    with conn.cursor() as cur:
        cur.execute(sql)

def main():
    conn = pymysql.connect(
        host="localhost",
        user="root",
        password=password,
        database="electricity_capstone",
        local_infile=1,
        autocommit=True
    )

    try:
        for chunk in sorted(CHUNKS.glob("*.csv")):
            print(f"Loading {chunk.name} ...")
            load_chunk(conn, chunk)
            conn.commit()
            print(f"✓ Loaded {chunk.name}")
    except Exception as e:
        conn.rollback()
        print("❌ Error — rolled back:", e)
    finally:
        conn.close()

if __name__ == "__main__":
    main()



Enter your password:  ········


Loading unpivot_humidity.part1.csv ...
✓ Loaded unpivot_humidity.part1.csv
Loading unpivot_humidity.part2.csv ...
✓ Loaded unpivot_humidity.part2.csv
Loading unpivot_humidity.part3.csv ...
✓ Loaded unpivot_humidity.part3.csv
Loading unpivot_humidity.part4.csv ...
✓ Loaded unpivot_humidity.part4.csv
Loading unpivot_pressure.part1.csv ...
✓ Loaded unpivot_pressure.part1.csv
Loading unpivot_pressure.part2.csv ...
✓ Loaded unpivot_pressure.part2.csv
Loading unpivot_pressure.part3.csv ...
✓ Loaded unpivot_pressure.part3.csv
Loading unpivot_pressure.part4.csv ...
✓ Loaded unpivot_pressure.part4.csv
Loading unpivot_temperature.part1.csv ...
✓ Loaded unpivot_temperature.part1.csv
Loading unpivot_temperature.part2.csv ...
✓ Loaded unpivot_temperature.part2.csv
Loading unpivot_temperature.part3.csv ...
✓ Loaded unpivot_temperature.part3.csv
Loading unpivot_temperature.part4.csv ...
✓ Loaded unpivot_temperature.part4.csv
Loading unpivot_weather_description.part1.csv ...
✓ Loaded unpivot_weather_d